[← All benchmarks](Evaluate_Cosmos3.ipynb)

# Perception Test

Understand actions, objects, and events using the public validation questions and local labels.

Choose smoke or full scope below, then select **Run → Run All Cells**.


In [ ]:
from notebook_ui import show_plan, show_results, invoke

# This page always runs Perception Test. Use All benchmarks to switch.
BENCHMARK = "perception-test"


## 01 / Configure the run

Keep `SCOPE = "smoke"` for a quick pipeline check; choose `"full"` for the complete Perception Test benchmark. Each attempt gets a fresh timestamped results folder.


In [ ]:
import json
import os
from pathlib import Path
import sys
from datetime import datetime, timezone
from benchmarks import validate_selection

SCOPE = os.environ.get("EVAL_SCOPE", "smoke")  # "smoke" or "full"
DOWNLOAD_PUBLIC = True  # False only when the selected pinned inputs are staged.
API_NPROC = 1  # 1–4 simultaneous API workers; do not run concurrent notebooks.

# Advanced: rescore existing Perception Test predictions without a model/video.
SCORE_ONLY = os.environ.get("EVAL_SCORE_ONLY") == "1"
EXISTING_WORK = None
PREDICTIONS_FILE = None  # Set exactly one existing source when SCORE_ONLY is True.

# Persistent mounts are separate from source. Override these for another runtime.
DATA_ROOT = Path(os.environ.get("BENCHMARK_DATA_DIR", "/datasets"))
RESULTS_ROOT = Path(os.environ.get("EVAL_RESULTS_DIR", "/results"))
DATA = (Path(os.environ.get("PERCEPTION_TEST_DATA_DIR", "/perception-test"))
        if BENCHMARK == "perception-test" else DATA_ROOT / BENCHMARK)
LAUNCHABLE = Path.cwd()
CLI = [sys.executable, "-u", str(LAUNCHABLE / "benchmarks.py")]
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
WORK = RESULTS_ROOT / BENCHMARK / f"{SCOPE}-{'score-' if SCORE_ONLY else ''}{STAMP}"
LOG = WORK.with_suffix(".log")

validate_selection(BENCHMARK, SCOPE, SCORE_ONLY)
if not 1 <= API_NPROC <= 4:
    raise ValueError("Choose API_NPROC between 1 and 4.")
if SCORE_ONLY and (EXISTING_WORK is None) == (PREDICTIONS_FILE is None):
    raise ValueError("Set exactly one of EXISTING_WORK or PREDICTIONS_FILE.")
if not (LAUNCHABLE / "benchmarks.py").is_file():
    raise ValueError("Run this notebook from its launchable directory.")
show_plan(BENCHMARK, SCOPE, DATA, WORK, API_NPROC, SCORE_ONLY)


## 02 / Download the public inputs

Downloads use pinned publisher URLs and checksums, then safe extraction. Existing verified assets are reused. **Check the download budget above before running:** a small smoke can still require a large source archive, especially MMSI-Video-Bench. No Hugging Face token or private NVIDIA dataset is required.

Inputs stay on your instance. Keep publisher attribution and applicable license terms; public access is not permission to redistribute every underlying video. For Perception Test score-only mode, this cell fetches metadata only.


In [ ]:
if DOWNLOAD_PUBLIC:
    command = CLI + ["download", "--benchmark", BENCHMARK,
                     "--scope", SCOPE, "--data-root", str(DATA)]
    if SCORE_ONLY:
        command += ["--metadata-only"]
    invoke(command, WORK.with_suffix(".download.log"))
else:
    print("Downloads skipped. The runner will validate the selected staged inputs before inference.")


## 03 / Run inference and score locally

This cell checks selected media/annotation coverage before model requests, streams real progress below, and writes a durable log. It then applies the selected benchmark’s local scorer. **No external scoring submission, external judge, or manual benchmark command.**

**Perception Test protocol:** 16 source-cut-aware sampled images; local top1 on public validation labels.

Run one benchmark at a time. Full runs can take hours. Interrupting leaves diagnostics; rerun the configuration cell before a new attempt to get a fresh output folder.


In [ ]:
command = CLI + ["run", "--benchmark", BENCHMARK, "--scope", SCOPE,
                 "--data-root", str(DATA), "--work-dir", str(WORK),
                 "--api-nproc", str(API_NPROC)]
if SCORE_ONLY:
    command += ["--score-only"]
    command += (["--existing-work", str(EXISTING_WORK)] if EXISTING_WORK is not None
                else ["--predictions", str(PREDICTIONS_FILE)])
invoke(command, LOG)


## 04 / Read the result, retain the evidence

The heading below is checked against the result artifact’s benchmark and scope. Accuracy and correlation are not the same metric; do not combine scores across benchmarks. **Smoke results are not full benchmark results.**

Expand the protocol record to inspect the exact setting. JSON reports, prediction evidence, source/dataset revisions, and logs remain in the benchmark’s own result folder. Links below are private runtime artifacts—not files to commit or publish.


In [ ]:
SUMMARY = WORK / ("notebook-summary.json" if SCORE_ONLY else "summary.json")
summary = json.loads(SUMMARY.read_text())
show_results(summary, BENCHMARK, SCOPE, WORK, LOG)


## Reviewer checklist / Before sharing with Dan

- Confirm the selected dataset, split, frame/pixel budget, generation settings, and primary scorer are the intended protocol.
- Inspect coverage, invalid/truncated answers, source/data/model revisions, and the retained logs. A matching score alone does not establish reproduction of a published result.
- To switch benchmarks, use [← All benchmarks](Evaluate_Cosmos3.ipynb) and open another notebook. Finish or interrupt this run first; closing its tab does not stop its kernel. For a new attempt here, use **Restart Kernel and Run All Cells**. Verified inputs are cached; new results are isolated.
- Share code and a reviewed aggregate result, not videos, labels, raw predictions, endpoint credentials, or executed notebook outputs. Clear all outputs before committing.

### Stop safely

In the **host terminal**, `docker compose --profile local stop` preserves mounted data/results. The existing evaluator service is still named `perception-eval` and the Compose project `cosmos3-perception-test`; both serve all three benchmark notebooks. Stopping containers does **not** stop Brev VM billing—stop the instance separately when finished. This notebook targets native Linux x86-64 on Brev, with the pinned Cosmos3-Nano endpoint or a configured compatible endpoint.
